In [6]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from scipy.stats import norm



In [7]:

# ---------------------------
# Load data (8D inputs)
# ---------------------------
X = np.load("initial_inputs.npy")     # shape (N, 8)
y = np.load("initial_outputs.npy")    # shape (N,)

# Ensure correct shapes
X = X.reshape(-1, 8)
y = y.reshape(-1)

# ---------------------------
# Fit Gaussian Process model
# ---------------------------
d = 8
kernel = (
    C(1.0, (1e-3, 1e3)) *
    RBF(length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3))
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-3))
)

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)

# ---------------------------
# Expected Improvement (EI) - returns 1D array
# ---------------------------
def expected_improvement(X_cand, gp, y_best, xi=1e-3):
    """
    X_cand: (M, d)
    returns: ei (M,)  -- one EI value per candidate
    """
    mu, sigma = gp.predict(X_cand, return_std=True)
    # ensure 1D arrays
    mu = np.asarray(mu).ravel()
    sigma = np.asarray(sigma).ravel()

    # numerical stability: avoid zero or negative sigma
    sigma = np.maximum(sigma, 1e-9)

    imp = y_best - mu - xi            # for minimisation
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei  # shape (M,)

# ---------------------------
# Generate candidate points in 8D
# ---------------------------
lower = X.min(axis=0)
upper = X.max(axis=0)

n_candidates = 25000
X_cand = np.random.uniform(lower, upper, size=(n_candidates, d))

# ---------------------------
# Compute EI and select next point (robustly)
# ---------------------------
y_best = np.min(y)    # minimisation
EI = expected_improvement(X_cand, gp, y_best)

# Mask non-finite EI values
finite_mask = np.isfinite(EI)
if not np.any(finite_mask):
    raise ValueError("All EI values are non-finite (NaN/Inf). Check GP predictions or candidate generation.")

# pick best index among finite values
finite_indices = np.nonzero(finite_mask)[0]
best_finite_idx = finite_indices[np.argmax(EI[finite_mask])]
next_point = X_cand[best_finite_idx]

print("Next proposed query point (8D):", next_point)


Next proposed query point (8D): [0.98366082 0.87842028 0.94937038 0.87399609 0.5074947  0.65851097
 0.8973769  0.0696678 ]


/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/stephensefa/anaconda3/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
